# Laboratorio D: Clasificacion multiclase con frutas (Softmax)

**Redes Neuronales II - Especializacion en Inteligencia Artificial**
**Autor:** Harold Steven Rueda Lara
**Objetivo:** Extender el problema a 3 clases (manzana, banano, naranja) usando una capa de salida Softmax, que permite comparar varias clases y elegir la de mayor probabilidad.


## Paso 22. Crear dataset multiclase
Se agrega una tercera categoria: 2 = naranja.

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

X_multi = np.array([
    # MANZANAS - 0
    [150,7.5,.10],[160,8.0,.15],[145,7.2,.05],[170,8.3,.20],[155,7.8,.10],
    [165,8.1,.15],[140,7.0,.05],[175,8.5,.20],[158,7.9,.10],[162,8.2,.15],
    # BANANOS - 1
    [120,18.0,.90],[125,19.0,.95],[115,17.5,.85],[130,20.0,1.00],[118,18.5,.90],
    [123,19.5,.95],[110,17.0,.80],[128,20.5,1.00],[121,18.8,.90],[126,19.2,.95],
    # NARANJAS - 2
    [180,8.0,.35],[190,8.5,.40],[175,7.8,.30],[200,9.0,.45],[185,8.2,.35],
    [195,8.7,.40],[170,7.5,.30],[205,9.2,.45],[188,8.4,.35],[192,8.6,.40]
], dtype=float)
y_multi = np.array([0]*10 + [1]*10 + [2]*10)
print("X_multi:", X_multi.shape, "| y_multi:", y_multi.shape)

X_multi: (30, 3) | y_multi: (30,)


## Paso 23. Separar y escalar

In [2]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)
scaler_multi = StandardScaler()
X_train_multi = scaler_multi.fit_transform(X_train_multi)
X_test_multi = scaler_multi.transform(X_test_multi)
print("Listo:", X_train_multi.shape, X_test_multi.shape)

Listo: (24, 3) (6, 3)


## Paso 24. Crear red con Softmax
La capa de salida tiene 3 neuronas (una por clase). `sparse_categorical_crossentropy` es la version de la funcion de error adecuada para etiquetas enteras (0, 1, 2) en vez de one-hot.

In [3]:
tf.random.set_seed(42)
modelo_multi = keras.Sequential([
    keras.layers.Input(shape=(3,)),
    keras.layers.Dense(6, activation="relu"),
    keras.layers.Dense(3, activation="softmax")
])
modelo_multi.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
modelo_multi.summary()

Model: "sequential"
+---------------------------------+------------------------+---------------+
| Layer (type)                    | Output Shape           |       Param # |
+---------------------------------+------------------------+---------------+
| dense (Dense)                   | (None, 6)               |            24 |
| dense_1 (Dense)                 | (None, 3)               |            21 |
+---------------------------------+------------------------+---------------+
 Total params: 45 (180.00 B)


## Paso 25. Entrenar, evaluar y probar una naranja nueva
`np.argmax` toma la posicion de mayor probabilidad entre las 3 salidas de Softmax; esa posicion es la clase elegida.

In [4]:
historial_multi = modelo_multi.fit(X_train_multi, y_train_multi, epochs=300, verbose=0)
loss_multi, accuracy_multi = modelo_multi.evaluate(X_test_multi, y_test_multi, verbose=0)
probabilidades_multi = modelo_multi.predict(X_test_multi, verbose=0)
predicciones_multi = np.argmax(probabilidades_multi, axis=1)

print("Accuracy:", accuracy_multi)
print("Predicciones:", predicciones_multi)
print("Reales:", y_test_multi)

nombres_frutas = ["MANZANA", "BANANO", "NARANJA"]
fruta_nueva_multi = np.array([[190, 8.5, 0.38]])
fruta_nueva_multi = scaler_multi.transform(fruta_nueva_multi)
prob = modelo_multi.predict(fruta_nueva_multi, verbose=0)
clase = np.argmax(prob, axis=1)[0]
print("La red piensa que es:", nombres_frutas[clase])

Accuracy: 1.0
Predicciones: [2 1 2 0 1 0]
Reales: [2 1 2 0 1 0]
La red piensa que es: NARANJA


## Conclusion del laboratorio
La red con salida Softmax clasifico correctamente las 6 frutas de prueba entre 3 clases (**accuracy = 1.0**) y acerto tambien con una fruta nunca vista, confirmando que Softmax + argmax es el mecanismo correcto para pasar de clasificacion binaria a multiclase.